In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break

SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai


In [2]:
import json
from collections import Counter

import pandas as pd

from app.infrastructure.pdf_loader import PDFLoader
from app.repositories.document_repository import FileDocumentRepository
from app.services.text_cleaning_service import TextCleaningService

In [3]:
pdf_files = sorted(RAW_DIR.rglob("*.pdf"))

In [4]:
loader = PDFLoader()
repo = FileDocumentRepository()
cleaner = TextCleaningService()

In [5]:
records = []

for pdf_path in pdf_files:
    loaded = loader.load(
        str(pdf_path),
        source="gesetze-im-internet",
        document_type="law",
        topic="healthcare-law",
    )

    cleaned_text = cleaner.clean(loaded.text)

    records.append(
        {
            "file_name": pdf_path.name,
            "page_count": loaded.metadata.extra.get("page_count"),
            "file_size_bytes": loaded.metadata.extra.get("file_size_bytes"),
            "raw_text_length": len(loaded.text),
            "clean_text_length": len(cleaned_text),
            "source": loaded.metadata.source,
            "document_type": loaded.metadata.document_type,
            "topic": loaded.metadata.topic,
        }
    )

df = pd.DataFrame(records)
df

,file_name,page_count,file_size_bytes,raw_text_length,clean_text_length,source,document_type,topic
0,PUEG_RefE_Pflegereform_bf.pdf,112,1301360,359167,355150,gesetze-im-internet,law,healthcare-law
1,SGB_11.pdf,177,873576,785894,784738,gesetze-im-internet,law,healthcare-law
2,SGB_5.pdf,568,2658246,2662222,2658704,gesetze-im-internet,law,healthcare-law


In [6]:
df.describe(include="all")

,file_name,page_count,file_size_bytes,raw_text_length,clean_text_length,source,document_type,topic
count,3,3.000000,3.000000e+00,3.000000e+00,3.000000e+00,3,3,3
unique,3,NaN,NaN,NaN,NaN,1,1,1
top,PUEG_RefE_Pflegereform_bf.pdf,NaN,NaN,NaN,NaN,gesetze-im-internet,law,healthcare-law
freq,1,NaN,NaN,NaN,NaN,3,3,3
mean,NaN,285.666667,1.611061e+06,1.269094e+06,1.266197e+06,NaN,NaN,NaN
std,NaN,246.658333,9.317712e+05,1.225205e+06,1.224926e+06,NaN,NaN,NaN
min,NaN,112.000000,8.735760e+05,3.591670e+05,3.551500e+05,NaN,NaN,NaN
25%,NaN,144.500000,1.087468e+06,5.725305e+05,5.699440e+05,NaN,NaN,NaN
50%,NaN,177.000000,1.301360e+06,7.858940e+05,7.847380e+05,NaN,NaN,NaN
75%,NaN,372.500000,1.979803e+06,1.724058e+06,1.721721e+06,NaN,NaN,NaN


In [7]:
df[["file_name", "page_count", "raw_text_length", "clean_text_length"]]

,file_name,page_count,raw_text_length,clean_text_length
0,PUEG_RefE_Pflegereform_bf.pdf,112,359167,355150
1,SGB_11.pdf,177,785894,784738
2,SGB_5.pdf,568,2662222,2658704


In [8]:
df["chars_per_page"] = df["clean_text_length"] / df["page_count"]
df[["file_name", "page_count", "clean_text_length", "chars_per_page"]]

,file_name,page_count,clean_text_length,chars_per_page
0,PUEG_RefE_Pflegereform_bf.pdf,112,355150,3170.982143
1,SGB_11.pdf,177,784738,4433.548023
2,SGB_5.pdf,568,2658704,4680.816901


In [9]:
sample_text = cleaner.clean(loader.load(
    str(pdf_files[0]),
    source="gesetze-im-internet",
    document_type="law",
    topic="healthcare-law",
).text)

words = sample_text.split()
word_counts = Counter(words)

print("Top 30 words:")
for word, count in word_counts.most_common(30):
    print(word, count)

Top 30 words:
der 2144
die 1697
- 1370
und 1279
in 778
des 693
für 607
den 572
Absatz 566
zu 552
§ 549
von 526
1 439
nach 405
wird 374
im 365
Satz 351
dem 332
auf 322
mit 307
zur 306
Die 268
eine 266
oder 247
Zu 241
sich 237
das 222
2 212
bei 210
durch 204


In [10]:
summary = pd.DataFrame(
    {
        "file_name": df["file_name"],
        "page_count": df["page_count"],
        "clean_text_length": df["clean_text_length"],
        "chars_per_page": df["chars_per_page"].round(2),
    }
)

summary

,file_name,page_count,clean_text_length,chars_per_page
0,PUEG_RefE_Pflegereform_bf.pdf,112,355150,3170.98
1,SGB_11.pdf,177,784738,4433.55
2,SGB_5.pdf,568,2658704,4680.82


In [11]:
recommendation = {
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "notes": [
        "Legal documents are long and structured.",
        "Chunking should preserve section continuity.",
        "Recursive character splitting is a good first approach.",
    ],
}

print(json.dumps(recommendation, indent=2, ensure_ascii=False))

{
  "chunk_size": 1000,
  "chunk_overlap": 200,
  "notes": [
    "Legal documents are long and structured.",
    "Chunking should preserve section continuity.",
    "Recursive character splitting is a good first approach."
  ]
}
